# Joint Baseline Strategy Diagnostics

Run HAPPO through `scripts/diagnose_joint_happo.py` and heuristic approaches through `scripts/diagnose_joint_baseline_strategies.py`, or provide an existing `FILELIST` of diagnostic JSON files to skip rerunning diagnostics. The notebook compares the same core joint performance metrics used for model comparison:

- scout recall: mean and std
- confirmation recall: mean and std
- full-confirm success: rate and binary std
- final confidence: mean and std
- final coverage: mean and std
- scout, confirmation, confidence, and coverage AUC
- time to 100%, 80%, and 50% confirmation recall
- scout-to-confirm latency

Edit the strategy settings in the first code cell. If `FILELIST` is empty, the notebook checks one expected JSON per strategy and runs only the missing diagnostics, with up to `MAX_PARALLEL_RUNS` approaches running concurrently. There is no communication-dropout sweep. If `FILELIST` is non-empty, it reads those files directly and does not launch diagnostics.


In [ ]:
# Baseline comparison configuration. Set CKPT to a HAPPO models directory when you want
# the heuristic baselines to reuse that checkpoint's scenario settings.
MODLABEL = 'uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_to_10'
CKPT = "results/harl_runs/wildfire/wildfire_search/happo/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_10/seed-00001-2026-07-19-00-15-35/models"
NSTEPS_PER_EPISODE = 900
SEED_START = 1000
SEED_END = 1099

# One diagnostic is produced for each strategy. This is not a dropout sweep.
STRATEGIES = ['happo', 'ant_colony', 'lawnmower', 'random_walk']
COMMS_DROPOUT = 0.0

NSURVIVORS_OBS = 10
NSURVIVORS_MIN = 5
NSURVIVORS_MAX = 10

RUN_DIAGNOSTICS = True
FORCE_RERUN_DIAGNOSTICS = False
# Use 1 for sequential execution; increase carefully because each diagnostic is CPU intensive.
MAX_PARALLEL_RUNS = 1

# Optional extra CLI arguments for the selected HAPPO/baseline diagnostic command.
# Examples: ['--disable-fire'], ['--enable-fire'], ['--n-uavs', '3'].
EXTRA_DIAGNOSTIC_ARGS = []

# Optional: provide existing JSON outputs to skip all diagnostic execution.
FILELIST = []
# FILELIST = [
#   'outputs/baseline_comparison/happo_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_to_10_dropout0p0_seeds_1000_1099.json',
#   'outputs/baseline_comparison/ant_colony_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_to_10_dropout0p0_seeds_1000_1099.json',
#   'outputs/baseline_comparison/lawnmower_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_to_10_dropout0p0_seeds_1000_1099.json',
#   'outputs/baseline_comparison/random_walk_uav4_ugv3_area_1sqkm_malibu_grid256_steps900_fire_survivors_5_to_10_dropout0p0_seeds_1000_1099.json',
# ]

# Optional display labels for FILELIST entries. Leave empty to derive labels from metadata.
LABELS = []


In [ ]:
import json
import shlex
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HAPPO_SCRIPT = Path('scripts/diagnose_joint_happo.py')
BASELINE_SCRIPT = Path('scripts/diagnose_joint_baseline_strategies.py')
CWD = Path.cwd().resolve()
if all((CWD / script).is_file() for script in (HAPPO_SCRIPT, BASELINE_SCRIPT)):
    PROJECT_ROOT = CWD
elif all((CWD.parent / script).is_file() for script in (HAPPO_SCRIPT, BASELINE_SCRIPT)):
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = Path('..').resolve()

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'baseline_comparison'
RESULTS_ROOT = PROJECT_ROOT / 'results/harl_runs/wildfire/wildfire_search/happo'

STRATEGY_LABELS = {
    'happo': 'OmniSearch RL',
    'ant_colony': 'ACO',
    'lawnmower': 'Lawnmower',
    'random_walk': 'Random Walk',
    'random_action': 'Random Action',
    'highest_confidence': 'Highest Confidence',
}


def _strategy_tag(strategy):
    normalized = str(strategy).strip().lower().replace('-', '_').replace(' ', '_')
    for tag in STRATEGY_LABELS:
        if normalized == tag or normalized.startswith(f'{tag}_'):
            return tag
    return normalized


def _diagnostic_prefix(strategy, checkpoint):
    strategy_tag = _strategy_tag(strategy)
    if strategy_tag == 'happo':
        return [
            sys.executable,
            str(PROJECT_ROOT / HAPPO_SCRIPT),
            '--checkpoint-dir', str(checkpoint),
        ]
    return [
        sys.executable,
        str(PROJECT_ROOT / BASELINE_SCRIPT),
        '--strategy', strategy_tag,
        '--happo-checkpoint', str(checkpoint),
    ]


def _resolve_checkpoint(ckpt):
    if ckpt is not None:
        path = Path(ckpt).expanduser()
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.is_dir():
            raise FileNotFoundError(f'Checkpoint directory not found: {path}')
        return path.resolve()

    candidates = sorted((RESULTS_ROOT / MODLABEL).glob('seed-*/models'), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f'No checkpoint found under {RESULTS_ROOT / MODLABEL}. Set CKPT explicitly.'
        )
    return candidates[-1].resolve()


#def _comparison_label():
#    label = MODLABEL.removeprefix('happo_')
#    trained_tag = f'survivors_{NSURVIVORS_OBS}'
#    active_tag = f'survivors_{NSURVIVORS_MIN}_to_{NSURVIVORS_MAX}'
#    return label.replace(trained_tag, active_tag)


def _diagnostic_output_paths(strategy):
    strategy_tag = str(strategy).replace('-', '_')
    dropout_tag = f'{float(COMMS_DROPOUT):.1f}'
    dropout_tag = dropout_tag.replace(".","p")
    stem = (
        f'{strategy_tag}_{MODLABEL}_dropout{dropout_tag}_'
        f'seeds_{SEED_START}_{SEED_END}'
    )
    return OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.png'


def _resolve_json_file(path):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([PROJECT_ROOT / path, PROJECT_ROOT / 'notebooks' / path])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    return path.resolve() if path.is_absolute() else (PROJECT_ROOT / path).resolve()


In [ ]:
def _run_diagnostic_job(strategy, cmd, log_path):
    with log_path.open('w') as log:
        subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            check=True,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    return strategy


if len(STRATEGIES) != len(set(STRATEGIES)):
    raise ValueError('STRATEGIES contains duplicates')
if int(MAX_PARALLEL_RUNS) < 1:
    raise ValueError('MAX_PARALLEL_RUNS must be at least 1')

if FILELIST:
    JSON_FILES = [_resolve_json_file(path) for path in FILELIST]
    print('Using provided file list; diagnostics will not be run.')
else:
    checkpoint = _resolve_checkpoint(CKPT)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    seeds = [str(seed) for seed in range(SEED_START, SEED_END + 1)]
    JSON_FILES = []
    jobs = []

    print('Checkpoint:', checkpoint)
    print('Strategies:', ', '.join(STRATEGIES))
    for index, strategy in enumerate(STRATEGIES, start=1):
        json_output, plots_output = _diagnostic_output_paths(strategy)
        JSON_FILES.append(json_output)
        should_run = RUN_DIAGNOSTICS and (
            FORCE_RERUN_DIAGNOSTICS or not json_output.is_file()
        )
        if not should_run:
            print(f'[{index}/{len(STRATEGIES)}] Using existing diagnostics:', json_output)
            continue

        cmd = _diagnostic_prefix(strategy, checkpoint)
        cmd.extend([
            '--steps', str(NSTEPS_PER_EPISODE),
            '--seeds', *seeds,
            '--n-survivors', str(NSURVIVORS_OBS),
            '--active-survivors-min', str(NSURVIVORS_MIN),
            '--active-survivors-max', str(NSURVIVORS_MAX),
            '--json-output', str(json_output),
            '--plots-output', str(plots_output),
        ])
        cmd.extend(str(arg) for arg in EXTRA_DIAGNOSTIC_ARGS)
        log_path = json_output.with_suffix('.log')
        jobs.append((strategy, cmd, log_path))
        print(f'[{index}/{len(STRATEGIES)}] Queued (~9h):', shlex.join(cmd))
        print('Log:', log_path)

    if jobs:
        workers = min(int(MAX_PARALLEL_RUNS), len(jobs))
        print(f'Running {len(jobs)} missing diagnostic(s) with {workers} worker(s).')
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(_run_diagnostic_job, strategy, cmd, log_path): (strategy, log_path)
                for strategy, cmd, log_path in jobs
            }
            for future in as_completed(futures):
                strategy, log_path = futures[future]
                future.result()
                print(f'Finished strategy={strategy}; log={log_path}')

missing = [path for path in JSON_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing diagnostic JSON files:\n' + '\n'.join(str(path) for path in missing))

print(f'Loaded file list: {len(JSON_FILES)} JSON file(s)')
for path in JSON_FILES:
    print(' -', path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path)


In [ ]:
def _summary_value(summary, *keys, default=float('nan')):
    for key in keys:
        if key in summary:
            return summary[key]
    return default


def _payload_scenario(payload):
    scenario = payload.get('scenario')
    if isinstance(scenario, dict) and scenario:
        return scenario
    scenario_kwargs = payload.get('scenario_kwargs')
    if isinstance(scenario_kwargs, dict) and scenario_kwargs:
        return scenario_kwargs
    metadata = payload.get('metadata', {})
    scenario_kwargs = metadata.get('scenario_kwargs') if isinstance(metadata, dict) else None
    return scenario_kwargs if isinstance(scenario_kwargs, dict) else {}


def _strategy_from_payload(path, payload):
    metadata = payload.get('metadata', {})
    strategy = metadata.get('strategy') if isinstance(metadata, dict) else None
    if isinstance(strategy, dict):
        return strategy.get('label') or strategy.get('name') or path.stem
    summary_strategy = payload.get('summary', {}).get('strategy')
    return summary_strategy or path.stem


def _success_rate(summary):
    rate = _summary_value(summary, 'full_confirm_success_rate', 'success_rate')
    if pd.isna(rate):
        percent = _summary_value(summary, 'full_confirm_success_percent')
        rate = percent / 100.0 if not pd.isna(percent) else float('nan')
    return rate


def _success_std(summary, rate):
    if pd.isna(rate):
        return float('nan')
    episodes = _summary_value(summary, 'episodes')
    if pd.isna(episodes) or episodes <= 0:
        return float('nan')
    return float(np.sqrt(max(rate * (1.0 - rate), 0.0)))


def _label_from_payload(path, payload):
    strategy = _strategy_from_payload(path, payload)
    return STRATEGY_LABELS.get(_strategy_tag(strategy), str(strategy).replace('_', ' ').title())


def _finite(values):
    return [float(value) for value in values if isinstance(value, (int, float, np.floating)) and np.isfinite(value)]


def _mean_std(values):
    values = _finite(values)
    if not values:
        return float('nan'), float('nan')
    return float(np.mean(values)), float(np.std(values))


def _episode_steps(row, scenario):
    for key in ('episode_steps', 'max_steps'):
        value = row.get(key)
        if isinstance(value, (int, float)) and value > 0:
            return int(value)
    value = scenario.get('max_steps')
    if isinstance(value, (int, float)) and value > 0:
        return int(value)
    return int(NSTEPS_PER_EPISODE)


def _active_survivor_indices(row):
    explicit = row.get('active_survivor_indices')
    if isinstance(explicit, list):
        return [int(idx) for idx in explicit]
    mask = row.get('active_survivor_mask')
    if isinstance(mask, list):
        return [idx for idx, active in enumerate(mask) if bool(active)]
    steps = row.get('first_scout_steps') or row.get('first_confirm_steps') or []
    count = row.get('active_survivors', row.get('survivors', len(steps)))
    try:
        count = int(count)
    except (TypeError, ValueError):
        count = len(steps)
    return list(range(min(count, len(steps))))


def _auc_from_first_steps(row, scenario, key):
    first_steps = row.get(key)
    if not isinstance(first_steps, list):
        return float('nan')
    steps = max(_episode_steps(row, scenario), 1)
    indices = _active_survivor_indices(row)
    if not indices:
        return 1.0
    total = 0.0
    for idx in indices:
        if idx >= len(first_steps):
            continue
        first_step = first_steps[idx]
        if first_step is None:
            continue
        try:
            first_step = float(first_step)
        except (TypeError, ValueError):
            continue
        if first_step <= 0:
            total += 1.0
        elif first_step <= steps:
            total += (steps - first_step + 1.0) / steps
    return float(total / max(len(indices), 1))


def _row_values(rows, *keys):
    values = []
    for row in rows:
        for key in keys:
            value = row.get(key)
            if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
                values.append(float(value))
                break
    return values


def _summary_or_rows(summary, rows, mean_key, std_key, *row_keys, fallback=None):
    mean = _summary_value(summary, mean_key)
    std = _summary_value(summary, std_key)
    if not pd.isna(mean):
        return float(mean), float(std) if not pd.isna(std) else float('nan')
    values = _row_values(rows, *row_keys)
    if not values and fallback is not None:
        values = [fallback(row) for row in rows]
    return _mean_std(values)


def _threshold_time(summary, threshold_key):
    container = summary.get('time_to_confirm_s', {})
    stats = container.get(threshold_key, {}) if isinstance(container, dict) else {}
    if not isinstance(stats, dict):
        stats = {}
    return {
        'mean': float(stats.get('mean_s', float('nan'))),
        'std': float(stats.get('std_s', float('nan'))),
        'reached_fraction': float(stats.get('reached_fraction', float('nan'))),
        'reached_count': float(stats.get('reached_count', float('nan'))),
    }


def _latency_stats(summary):
    fast = summary.get('fast_metrics', {})
    latency = fast.get('scout_to_confirm_latency_s') if isinstance(fast, dict) else None
    if isinstance(latency, dict):
        mean = latency.get('mean', float('nan'))
        std = latency.get('std', float('nan'))
    else:
        mean = _summary_value(summary, 'mean_scout_to_confirm_latency_s')
        std = _summary_value(summary, 'std_scout_to_confirm_latency_s')
    return float(mean), float(std)


if LABELS and len(LABELS) != len(JSON_FILES):
    raise ValueError(f'LABELS has {len(LABELS)} entries, but JSON_FILES has {len(JSON_FILES)} files')

records = []
for file_index, path in enumerate(JSON_FILES):
    payload = json.loads(path.read_text())
    summary = payload.get('summary', {})
    rows = payload.get('rows', [])
    scenario = _payload_scenario(payload)
    success_rate = _success_rate(summary)
    scout_auc_mean, scout_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_scout_auc',
        'std_scout_auc',
        'scout_auc',
        fallback=lambda row, scenario=scenario: _auc_from_first_steps(row, scenario, 'first_scout_steps'),
    )
    confirm_auc_mean, confirm_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_confirm_auc',
        'std_confirm_auc',
        'confirm_auc',
        'confirmation_auc',
        fallback=lambda row, scenario=scenario: _auc_from_first_steps(row, scenario, 'first_confirm_steps'),
    )
    coverage_auc_mean, coverage_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_coverage_auc',
        'std_coverage_auc',
        'coverage_auc',
    )
    confidence_auc_mean, confidence_auc_std = _summary_or_rows(
        summary,
        rows,
        'mean_confidence_auc',
        'std_confidence_auc',
        'confidence_auc',
    )
    confirm_100 = _threshold_time(summary, 'recall_100')
    confirm_080 = _threshold_time(summary, 'recall_080')
    confirm_050 = _threshold_time(summary, 'recall_050')
    latency_mean, latency_std = _latency_stats(summary)
    uav_path_length_mean, uav_path_length_std = _summary_or_rows(
        summary,
        rows,
        'mean_uav_path_length_m',
        'std_uav_path_length_m',
        'uav_path_length_m',
    )
    ugv_travel_cost_mean, ugv_travel_cost_std = _summary_or_rows(
        summary,
        rows,
        'mean_ugv_travel_cost_per_ground_step',
        'std_ugv_travel_cost_per_ground_step',
        'ugv_travel_cost_per_ground_step',
        'ugv_travel_cost_per_step',
    )
    records.append({
        'file': str(path),
        'label': str(LABELS[file_index]) if LABELS else _label_from_payload(path, payload),
        'strategy': _strategy_from_payload(path, payload),
        'episodes': _summary_value(summary, 'episodes', default=len(rows) if rows else float('nan')),
        'scout_recall_mean': _summary_value(summary, 'mean_scout_recall'),
        'scout_recall_std': _summary_value(summary, 'std_scout_recall'),
        'confirm_recall_mean': _summary_value(summary, 'mean_confirm_recall'),
        'confirm_recall_std': _summary_value(summary, 'std_confirm_recall'),
        'success_mean': success_rate,
        'success_std': _success_std(summary, success_rate),
        'success_count': _summary_value(summary, 'full_confirm_success_count'),
        'final_confidence_mean': _summary_value(summary, 'mean_final_confidence'),
        'final_confidence_std': _summary_value(summary, 'std_final_confidence'),
        'final_coverage_mean': _summary_value(summary, 'mean_final_coverage_fraction'),
        'final_coverage_std': _summary_value(summary, 'std_final_coverage_fraction'),
        'scout_auc_mean': scout_auc_mean,
        'scout_auc_std': scout_auc_std,
        'confirm_auc_mean': confirm_auc_mean,
        'confirm_auc_std': confirm_auc_std,
        'coverage_auc_mean': coverage_auc_mean,
        'coverage_auc_std': coverage_auc_std,
        'confidence_auc_mean': confidence_auc_mean,
        'confidence_auc_std': confidence_auc_std,
        'confirm_time_100_mean_s': confirm_100['mean'],
        'confirm_time_100_std_s': confirm_100['std'],
        'confirm_time_100_reached_fraction': confirm_100['reached_fraction'],
        'confirm_time_080_mean_s': confirm_080['mean'],
        'confirm_time_080_std_s': confirm_080['std'],
        'confirm_time_080_reached_fraction': confirm_080['reached_fraction'],
        'confirm_time_050_mean_s': confirm_050['mean'],
        'confirm_time_050_std_s': confirm_050['std'],
        'confirm_time_050_reached_fraction': confirm_050['reached_fraction'],
        'scout_to_confirm_latency_mean_s': latency_mean,
        'scout_to_confirm_latency_std_s': latency_std,
        'uav_path_length_mean_m': uav_path_length_mean,
        'uav_path_length_std_m': uav_path_length_std,
        'ugv_travel_cost_mean_per_ground_step': ugv_travel_cost_mean,
        'ugv_travel_cost_std_per_ground_step': ugv_travel_cost_std,
    })

metrics = pd.DataFrame.from_records(records)
metrics


## Compact Table

The table below formats each metric as `mean +/- std` for quick comparison. AUC values are time-integrated performance scores. Confirmation timing values are in minutes and are conditional on episodes that reached the requested confirmation recall threshold; the adjacent reached columns show how many episodes made it that far.


In [ ]:
def _pm(mean, std, *, digits=3, suffix=''):
    if pd.isna(mean):
        return 'n/a'
    if pd.isna(std):
        return f'{mean:.{digits}f}{suffix}'
    return f'{mean:.{digits}f} +/- {std:.{digits}f}{suffix}'


def _pct(value):
    if pd.isna(value):
        return 'n/a'
    return f'{100.0 * float(value):.0f}%'


compact = pd.DataFrame({
    'label': metrics['label'],
    'episodes': metrics['episodes'].astype('Int64'),
    'scout recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_recall_mean'], metrics['scout_recall_std'])
    ],
    'confirm recall': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_recall_mean'], metrics['confirm_recall_std'])
    ],
    'success': [
        _pm(mean, std)
        for mean, std in zip(metrics['success_mean'], metrics['success_std'])
    ],
    'success count': metrics['success_count'].astype('Int64'),
    'final confidence': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_confidence_mean'], metrics['final_confidence_std'])
    ],
    'final coverage': [
        _pm(mean, std)
        for mean, std in zip(metrics['final_coverage_mean'], metrics['final_coverage_std'])
    ],
    'confirm 100% time (min)': [
        _pm(mean, std, digits=1, suffix=' min')
        for mean, std in zip(metrics['confirm_time_100_mean_s'] / 60.0, metrics['confirm_time_100_std_s'] / 60.0)
    ],
    'confirm 100% reached': [_pct(value) for value in metrics['confirm_time_100_reached_fraction']],
    'confirm 80% time (min)': [
        _pm(mean, std, digits=1, suffix=' min')
        for mean, std in zip(metrics['confirm_time_080_mean_s'] / 60.0, metrics['confirm_time_080_std_s'] / 60.0)
    ],
    'confirm 80% reached': [_pct(value) for value in metrics['confirm_time_080_reached_fraction']],
    'confirm 50% time (min)': [
        _pm(mean, std, digits=1, suffix=' min')
        for mean, std in zip(metrics['confirm_time_050_mean_s'] / 60.0, metrics['confirm_time_050_std_s'] / 60.0)
    ],
    'confirm 50% reached': [_pct(value) for value in metrics['confirm_time_050_reached_fraction']],
    'scout-to-confirm latency (min)': [
        _pm(mean, std, digits=1, suffix=' min')
        for mean, std in zip(metrics['scout_to_confirm_latency_mean_s'] / 60.0, metrics['scout_to_confirm_latency_std_s'] / 60.0)
    ],
    'scout AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['scout_auc_mean'], metrics['scout_auc_std'])
    ],
    'confirm AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['confirm_auc_mean'], metrics['confirm_auc_std'])
    ],
    'coverage AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['coverage_auc_mean'], metrics['coverage_auc_std'])
    ],
    'confidence AUC': [
        _pm(mean, std)
        for mean, std in zip(metrics['confidence_auc_mean'], metrics['confidence_auc_std'])
    ],
}).set_index('label')

compact


## Plots

The notebook plots the time-integrated AUC metrics first, then a separate confirmation-timing plot. Timing bars are in minutes, so lower is better.


In [ ]:
PAPER_BG = '#f2ebd3ff'
GRID_COLOR = '#cfc7ad'
TEXT_GREEN = '#23562f'
TEXT_DARK = '#2f2a24'
AUC_METRIC_SPECS = [
    ('success', 'Success', '#8f2418', 's'),
    ('confirm_auc', 'Confirm AUC', '#c94a27', 's'),
    ('scout_auc', 'Scout AUC', '#e09b4f', 'o'),
    ('confidence_auc', 'Confidence AUC', '#1f5a35', '^'),
    ('coverage_auc', 'Coverage AUC', '#5f812f', 'v'),
]
TIME_METRIC_SPECS = [
    ('confirm_time_100', '100% Confirm', '#8f2418'),
    ('confirm_time_080', '80% Confirm', '#c94a27'),
    ('confirm_time_050', '50% Confirm', '#1f5a35'),
    ('scout_to_confirm_latency', 'Scout-to-Confirm', '#5f812f'),
]

plot_df = metrics.copy().reset_index(drop=True)
for prefix in (
    'confirm_time_100',
    'confirm_time_080',
    'confirm_time_050',
    'scout_to_confirm_latency',
):
    plot_df[f'{prefix}_mean_min'] = pd.to_numeric(plot_df[f'{prefix}_mean_s'], errors='coerce') / 60.0
    plot_df[f'{prefix}_std_min'] = pd.to_numeric(plot_df[f'{prefix}_std_s'], errors='coerce') / 60.0

plt.rcParams.update({
    'figure.facecolor': PAPER_BG,
    'axes.facecolor': PAPER_BG,
    'axes.edgecolor': '#f2ebd3ff',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 24,
    'axes.labelsize': 18,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'text.color': TEXT_DARK,
    'axes.labelcolor': TEXT_DARK,
    'xtick.color': TEXT_DARK,
    'ytick.color': TEXT_DARK,
    'legend.frameon': True,
    'font.family': 'serif',
})


def _available_specs(specs, df, *, mean_suffix='_mean'):
    keep = []
    for spec in specs:
        prefix = spec[0]
        col = f'{prefix}{mean_suffix}'
        if col in df.columns and pd.to_numeric(df[col], errors='coerce').notna().any():
            keep.append(spec)
    return keep


def _add_bar_labels(ax, bars, std, *, offset_rank=0, suffix='', digits=2, show_error=True):
    for bar, err in zip(bars, std):
        height = bar.get_height()
        if not np.isfinite(height):
            continue
        err = 0.0 if not np.isfinite(err) else float(err)
        label_err = err if show_error else 0.0
        label_text = f'{height:.{digits}f}{suffix}'
        if show_error:
            label_text = f'{height:.{digits}f}\u00b1{err:.{digits}f}{suffix}'
        ax.annotate(
            label_text,
            xy=(bar.get_x() + bar.get_width() / 2.0, height + label_err),
            xytext=(0, 22 + 5 * (offset_rank % 2)),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=16.5,
            fontweight='bold',
            rotation=90,
            linespacing=0.9,
            color=TEXT_DARK,
            clip_on=False,
        )


def _plot_grouped_bars(ax, df, specs, *, title, ylabel, mean_suffix='_mean', std_suffix='_std', y_limit=None, label_suffix='', label_digits=2):
    specs = _available_specs(specs, df, mean_suffix=mean_suffix)
    if not specs:
        ax.set_axis_off()
        ax.set_title(f'{title} not available', color=TEXT_GREEN, pad=12)
        return
    positions = np.arange(len(df))
    width = min(0.155, 0.76 / max(len(specs), 1))
    offset_units = np.arange(len(specs), dtype=float) - (len(specs) - 1) / 2.0
    series = []
    ymax = 1.0
    for prefix, label, color, *_rest in specs:
        mean = pd.to_numeric(df[f'{prefix}{mean_suffix}'], errors='coerce').to_numpy(dtype=float)
        std = pd.to_numeric(df[f'{prefix}{std_suffix}'], errors='coerce').to_numpy(dtype=float)
        plot_mean = np.where(np.isfinite(mean), mean, 0.0)
        show_error = prefix != 'success'
        plot_std = np.where(np.isfinite(std) & show_error, std, 0.0)
        if len(plot_mean):
            ymax = max(ymax, float(np.nanmax(plot_mean + plot_std)) * 1.45)
        series.append((prefix, label, color, mean, std, plot_mean, plot_std, show_error))
    if y_limit is None:
        ax.set_ylim(0.0, ymax)
    else:
        ax.set_ylim(*y_limit)
    for offset, (_prefix, label, color, mean, std, plot_mean, plot_std, show_error) in enumerate(series):
        bar_x = positions + offset_units[offset] * width
        bars = ax.bar(
            bar_x,
            plot_mean,
            width=width,
            yerr=plot_std,
            capsize=3 if show_error else 0,
            color=color,
            alpha=0.98,
            label=label,
            edgecolor=PAPER_BG,
            linewidth=0.8,
            error_kw={'elinewidth': 1.0, 'alpha': 0.65, 'ecolor': TEXT_DARK},
        )
        _add_bar_labels(ax, bars, std, offset_rank=offset, suffix=label_suffix, digits=label_digits, show_error=show_error)
    labels = df['label'].str.replace(r' \| .*$', '', regex=True).tolist()
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=0, ha='center', fontweight='bold', fontsize=16)
    ax.set_title(title, color=TEXT_GREEN, pad=12)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', color=GRID_COLOR, alpha=0.55, linewidth=0.9)
    ax.grid(axis='x', color=GRID_COLOR, alpha=0.30, linewidth=0.9)
    ax.spines['left'].set_color('#9d9279')
    ax.spines['bottom'].set_color('#9d9279')
    legend = ax.legend(
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
        title=None,
        fontsize=14,
    )
    legend.get_frame().set_facecolor(PAPER_BG)
    legend.get_frame().set_edgecolor('#c8bea4')
    legend.get_frame().set_alpha(0.90)


def _plot_single_metric_bars(
    ax,
    df,
    *,
    mean_column,
    std_column,
    scale,
    title,
    ylabel,
    color,
    label_digits,
):
    means = pd.to_numeric(df[mean_column], errors='coerce').to_numpy(dtype=float) * scale
    stds = pd.to_numeric(df[std_column], errors='coerce').to_numpy(dtype=float) * scale
    positions = np.arange(len(df))
    plot_means = np.where(np.isfinite(means), means, 0.0)
    plot_stds = np.where(np.isfinite(stds), stds, 0.0)
    upper = plot_means + plot_stds
    ymax = float(np.max(upper)) * 1.35 if len(upper) and np.max(upper) > 0 else 1.0
    bars = ax.bar(
        positions,
        plot_means,
        width=0.62,
        yerr=plot_stds,
        capsize=5,
        color=color,
        edgecolor=PAPER_BG,
        linewidth=0.9,
        error_kw={'elinewidth': 1.2, 'alpha': 0.70, 'ecolor': TEXT_DARK},
    )
    for bar, err in zip(bars, stds):
        height = bar.get_height()
        if not np.isfinite(height):
            continue
        err = float(err) if np.isfinite(err) else 0.0
        ax.annotate(
            f'{height:.{label_digits}f}±{err:.{label_digits}f}',
            xy=(bar.get_x() + bar.get_width() / 2.0, height + err),
            xytext=(0, 8),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=13.5,
            fontweight='bold',
            color=TEXT_DARK,
            clip_on=False,
        )
    labels = df['label'].str.replace(r' \| .*$', '', regex=True).tolist()
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontweight='bold', fontsize=15)
    ax.set_ylim(0.0, ymax)
    ax.set_title(title, color=TEXT_GREEN, pad=12)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', color=GRID_COLOR, alpha=0.55, linewidth=0.9)
    ax.spines['left'].set_color('#9d9279')
    ax.spines['bottom'].set_color('#9d9279')

fig, ax = plt.subplots(figsize=(max(18, len(plot_df) * 2.1), 6.3), facecolor=PAPER_BG)
_plot_grouped_bars(
    ax,
    plot_df,
    AUC_METRIC_SPECS,
    title='Success And Time-Integrated Performance (AUC)',
    ylabel='Score',
    y_limit=(0.0, 1.35),
    label_digits=2,
)
fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 6.3), facecolor=PAPER_BG)
_plot_single_metric_bars(
    axes[0],
    plot_df,
    mean_column='uav_path_length_mean_m',
    std_column='uav_path_length_std_m',
    scale=1.0 / 1000.0,
    title='UAV Path Length',
    ylabel='Total UAV path / episode (km)',
    color='#e09b4f',
    label_digits=1,
)
_plot_single_metric_bars(
    axes[1],
    plot_df,
    mean_column='ugv_travel_cost_mean_per_ground_step',
    std_column='ugv_travel_cost_std_per_ground_step',
    scale=1000.0,
    title='UGV Travel Cost',
    ylabel='Cost / UGV-step (x 10^-3)',
    color='#1f5a35',
    label_digits=2,
)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(max(18, len(plot_df) * 2.1), 6.3), facecolor=PAPER_BG)
_plot_grouped_bars(
    ax,
    plot_df,
    TIME_METRIC_SPECS,
    title='Confirmation Timing',
    ylabel='Minutes (lower is better)',
    mean_suffix='_mean_min',
    std_suffix='_std_min',
    label_suffix='',
    label_digits=1,
)
fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
plt.show()
